# Mouse vs lizard monocular rate and asymmetry

Exploratory comparison of **unique-gaze % monocular** and **how asymmetric** unpaired events are (fellow-eye residual from Fig 2f sampling).

**Definition:** a saccade is monocular if it is unpaired with a contralateral detected event within `binocular.sync_diff_ms` (**34 ms**).

**Cohorts:** lizard `configs/paper_blocks.yaml` (5 animals) vs mouse `configs/mouse_M_002_blocks.yaml` (**M_002 only**, 4 blocks).

**Primary filters:** all events, **no** head-stationary gate (mouse has none). Each species keeps its paper detector floor (lizard 0.8 °/frame, mouse 3.23 °/frame).

**What this can claim:** in these blocks, under the 34 ms unpaired definition, lizard vs M_002 differ (or do not) in unique-gaze % monocular and in fellow-eye residual when unpaired. Both vs their own shuffle nulls.

**What this cannot claim:** a well-powered species effect, or that any gap is independent of the detector floors without the sensitivity sweeps.

Output: `outputs/review_answers_latest/monocular_species_compare/` (`plots/`, `metadata/`, `LOGIC.md`, `replot.py`).

## 0. Setup

In [ ]:
%matplotlib inline
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.ioff()

REPO = Path.cwd()
if not (REPO / "src" / "eye_tracking_system_tools").is_dir():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "eye_tracking_system_tools").is_dir():
            REPO = p
            break

sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("MPLCONFIGDIR", str(REPO / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

from eye_tracking_system_tools.analysis.block_registry import load_registry
from eye_tracking_system_tools.analysis.event_cache import build_or_load_event_tables
from eye_tracking_system_tools.analysis.export_meta import load_params_yaml
from eye_tracking_system_tools.analysis.monocular_species import (
    export_monocular_species_compare,
)
from eye_tracking_system_tools.analysis.review_collect import review_answers_dir

# Empty TAG overwrites outputs/review_answers_latest
TAG = ""
LIZARD_REGISTRY = REPO / "configs" / "paper_blocks.yaml"
LIZARD_PARAMS = REPO / "configs" / "analysis_params.yaml"
MOUSE_REGISTRY = REPO / "configs" / "mouse_M_002_blocks.yaml"
MOUSE_PARAMS = REPO / "configs" / "analysis_params_mouse.yaml"

run = review_answers_dir(REPO / "outputs", TAG)
CACHE_DIR = run / "metadata" / "event_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("REPO:", REPO)
print("run :", run)
print("cache:", CACHE_DIR)

## 1. Load lizard and mouse event tables

Events only (`keep_traces=False`). The exporter reloads traces for the monocular Fig 2f / asymmetry step.

In [ ]:
lizard_params = load_params_yaml(LIZARD_PARAMS)
lizard_specs = load_registry(LIZARD_REGISTRY)
lizard_tables, lizard_cache, lizard_from_cache = build_or_load_event_tables(
    lizard_specs,
    lizard_params,
    CACHE_DIR,
    keep_traces=False,
    prefer_finalized=True,
)
print(
    f"lizard blocks={len(lizard_tables.blocks)} events={len(lizard_tables.all_saccades)} "
    f"cache={'hit' if lizard_from_cache else 'miss'}"
)
print(lizard_cache)

mouse_params = load_params_yaml(MOUSE_PARAMS)
mouse_specs = load_registry(MOUSE_REGISTRY)
mouse_tables, mouse_cache, mouse_from_cache = build_or_load_event_tables(
    mouse_specs,
    mouse_params,
    CACHE_DIR,
    keep_traces=False,
    prefer_finalized=True,
)
print(
    f"mouse  blocks={len(mouse_tables.blocks)} events={len(mouse_tables.all_saccades)} "
    f"cache={'hit' if mouse_from_cache else 'miss'}"
)
print(mouse_cache)

## 2. Export comparison

Writes unique-gaze % monocular (animal + block), shuffle nulls, pairing-window and relative-threshold sweeps, and monocular asymmetry index from Fig 2f fellow-eye sampling.

Set `n_shuffle=0` to skip the coincidence null. Default 200 is enough for a histogram. `collect_2f=False` skips AI / heatmaps (no trace reload).

In [ ]:
written = export_monocular_species_compare(
    lizard_tables,
    mouse_tables,
    run,
    show=True,
    n_shuffle=200,
    collect_2f=True,
)
for name, path in written.items():
    print(name, "→", path)

## 3. Summary tables

In [ ]:
meta = run / "monocular_species_compare" / "metadata"
summary = pd.read_csv(meta / "species_summary.csv")
counts = pd.read_csv(meta / "monocular_counts.csv")
display(summary)
display(counts.loc[counts["level"].isin(["animal", "cohort"])])
ai_path = meta / "monocular_ai_by_animal.csv"
if ai_path.is_file():
    display(pd.read_csv(ai_path))